# 데이터 전처리


### 라이브러리 설치


In [ ]:
!pip install torch==2.4.0 transformers==4.45.1 datasets==3.0.1 accekerate==0.34.2 trl==0.11.1 peft==0.13.0

In [1]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

c:\workspace\python\rag_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- 데이터셋 로딩 및 변환
  - load_dataset, Dataset: 다양한 데이터셋을 쉽게 로드하고 처리할 수 있습니다. load_dataset은 허깅페이스에 업로드된 데이터셋을 불러오거나 로컬 파일을 로드하는 데 사용하며, Dataset은 개별 데이터셋 객체를 다룰 때 사용합니다.
- 딥러닝 모델과 토크나이저
  - AutoModelForCausalLM: 허깅페이스 Transformers 라이브러리에서 제공하는 모델 다운로드를 위한 도구입니다. 언어 모델을 로드하는데 사용합니다.
  - AutoTokenizer: 특정 모델에 맞는 토크나이저를 자동으로 불러오는 도구입니다. 토크나이저는 텍스트를 언어 모델이 처리할 수 있는 정수 시퀀스로 변환하거나 정수 시퀀스를 다시 텍스트 문자열로 복원하는 역할을 합니다.
- 파인튜닝 및 효율적인 학습 구성
  - LoraConfig: 이번 실습에서 사용할 학습 방법인 LoRA(Low-Rank Adaptation) 학습 방식을 사용할 때 필요한 각종 설정값을 정의합니다. LoRA는 학습할 때 대규모 언어 모델 전체를 업데이트하는 것이 아니라, 대규모 언어 모델의 특정 부분만 업데이트하여 보다 효율적으로 학습하는 방식입니다.
- SFT(지도 학습 방식의 파인튜닝) 설정 및 학습도구
  - SFTConfig: 모델을 학습할 때 필요한 다양한 설정값을 정의하는 도구입니다. 학습과정에서 모델을 어떻게 업데이트할지 조정하며 여기에서는 학습률과 배치 크기, 옵티마이저등의 설정이 포함됩니다. 모델 전체를 학습할 때도 쓰일 수 있지만, 이번 실습에 사용하는 LoRA 학습에서처럼 모델의 특정 부분만 학습하는 방식에서도 사용합니다. SFTConfig에서 설정하는 모델의 학습 성능과 안정성에 큰 영향을 줍니다.
  - 예를들어 학습률이 너무 크면 학습이 불안정하고, 너무 작으면 속도가 느려집니다. 배치크기는 한번에 처리하는 데이터 개수를 결정하며 크기에 따라 학습 안정성과 메모리 사용량이 달라집니다. 옵티마이저는 모델을 업데이트하는 방식과 관련된 것으로, Adam SGD등 여러 종류가 있으며 학습 성능을 좌우합니다.
  - SFTConfig에는 이 외에도 가중치 감쇠, 학습 스케쥴링, 혼합 정밀도 학습(fp16) 등 학습에 영향을 주는 다양한 설정을 포함할 수 있습니다. LoRA를 적용하면 모델 전체가 아닌 일부 가중치만 학습하므로 LoRA와 연관된 설정은 LoraConfig에서 관리하지만, LoRA 학습과는 별개인 학습 과정 전반의 설정은 여전히 SFTConfig에서 관리합니다. 따라서 SFTConfig는 LoRAConfig와 함께 사용하며 학습을 조정하는 역할을 합니다.
- SFTTrainer: 실제 학습을 수행하는 클래스입니다. 주어진 데이터셋을 이용해 파일튜닝 과정을 자동으로 수행하며, 특정 부분만 업데이트하는 LoRA와 같은 학습 기봅도 적용할 수 있습니다. 모델, 데이터셋, 학습 설정을 한 번에 입력하여 효율적인 학습을 진행할 수 있도록 돕습니다.


In [2]:
# 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/klue-mrc-ko-rag-dataset", split="train")

# system_message 정의
system_message = """당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.

다음의 지시사항을 따르십시오.
1. 질문과 검색 결과를 바탕으로 답변하십시오.
2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.
3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다."라고 답변하신시오.
4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오.
  예를 들어 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.
5. 예를 들어 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]라고 기재하십시오.
6. 최대한 다수의 문서를 인용하여 답변하십시오.

검색 결과:
-----
{search_result}"""

# 원본 데이터의 type별 분포 출력
print("원본 데이터의 type 분포:")
for type_name in set(dataset["type"]):
    print(f"{type_name}: {dataset['type'].count(type_name)}개")

# train/test 분할 비율 설정(0.5면 5:5로 분할)
test_ratio = 0.8

train_data = []
test_data = []

# type별로 순회하면서 train/test 데이터 분할
for type_name in set(dataset["type"]):
    # 현재 type에 해당하는 데이터 인덱스만 추출
    curr_type_data = [i for i in range(len(dataset)) if dataset[i]["type"] == type_name]

    # test_ratio에 따라 test 데이터 개수 계산
    test_size = int(len(curr_type_data) * test_ratio)

    # 현재 type의 데이터를 test_ratio 비율로 분할하여 추가
    test_data.extend(curr_type_data[:test_size])
    train_data.extend(curr_type_data[test_size:])


# OpenAI format으로 데이터를 변환하기 위한 함수
def format_data(sample):
    # 검색 결과를 문서1, 문서2... 형태로 포매팅
    search_result = "\n-----\n".join(
        [
            f"문서{idx + 1}: {result}"
            for idx, result in enumerate(sample["search_result"])
        ]
    )

    # OpenAI forma로 변환
    return {
        "messages": [
            {
                "role": "system",
                "content": system_message.format(search_result=search_result),
            },
            {"role": "user", "content": sample["question"]},
            {"role": "assistant", "content": sample["answer"]},
        ]
    }


# 분할된 데이터를 OpenAI format으로 변환
train_dataset = [format_data(dataset[i]) for i in train_data]
test_dataset = [format_data(dataset[i]) for i in test_data]

# 최종 데이터셋 크기 출력
print(
    f"\n전체 데이터 분할 결과: Train: {len(train_dataset)}개, Test: {len(test_dataset)}개"
)

# 분할된 데이터의 type별 분포 출력
print("\n학습 데이터의 type 분포:")
for type_name in set(dataset["type"]):
    count = sum(1 for i in train_data if dataset[i]["type"] == type_name)
    print(f"{type_name}: {count}개")

print("\n테스트 데이터의 type 분포:")
for type_name in set(dataset["type"]):
    count = sum(1 for i in test_data if dataset[i]["type"] == type_name)
    print(f"{type_name}: {count}개")


원본 데이터의 type 분포:
no_answer: 404개
synthetic_question: 497개
paraphrased_question: 196개
mrc_question: 491개
mrc_question_with_1_to_4_negative: 296개

전체 데이터 분할 결과: Train: 380개, Test: 1504개

학습 데이터의 type 분포:
no_answer: 81개
synthetic_question: 100개
paraphrased_question: 40개
mrc_question: 99개
mrc_question_with_1_to_4_negative: 60개

테스트 데이터의 type 분포:
no_answer: 323개
synthetic_question: 397개
paraphrased_question: 156개
mrc_question: 392개
mrc_question_with_1_to_4_negative: 236개


1. 데이터셋 로드

- 허깅페이스 허브에서 데이터 셋을 불러옵니다. load_dataset() 함수를 사용하여 데이터셋을 불러옵니다.


2. 시스템 프롬프트 정의

- 학습에서 사용할 시스템 프롬프트를 정의합니다. 이 프롬프트에는 RAG 성능을 높이기 위한 여러가지 중요한 지침이 담겨있습니다.
- 첫째. 검색결과를 바탕으로 답변을 생성하도록 합니다.
- 둘째. 검색결과에 없는 내용으로 답변하지 말라는 제약을 둡니다.
- 셋째. 특정 질문에 대한 내용이 검색결과에 없을 경우 그 사실을 명시적으로 알리도록 합니다.
- 넷째. 답변시 참고한 문서는 반드시 [[ref1]]과 같은 형식으로 표시하도록 요구합니다. 여러 문서를 참고한 경우 [[ref1]], [[ref5]]와 같이 모든 참고 문서를 표시하도록 합니다.
- 마지막: 가능한 한 많은 문서를 참고하여 답변하도록 지시합니다.
- 프롬프트 끝의 {search_result}는 추후 실제 검색 결과로 대체됩니다. 학습할 때도 우리가 원하는 방향으로 대규모 언어 모델이 답변하도록 상세한 시스템프롬프트를 작성해야 합니다.


3. 원본 데이터 타입 분포 확인

- 원본 데이터의 type별 분포를 확인합니다.
- set() 함수로 중복없는 type 목록을 만들고, count() 메서드로 각 type이 몇 번 등장하는지 계산해 출력합니다.
- 데이터셋에는 앞에서 설명했듯이 synthetic_question, mrc_question, mrc_question_with_1_to_4_negative, paraphrased_question, no_answer라는 5가지 타입이 있습니다.


4. 학습용/테스트용 데이터 분할 비율 설정

- train/test 데이터 분할 비율을 설정합니다.
- test_ratio 변수에 0.8을 할당하여 전체 데이터의 80%를 테스트 데이터로, 나머지 20%를 학습데이터로 사용하도록 지정합니다.
- 보통은 학습 데이터의 양이 더 많고, 성능을 평가하기 위한 테스트 데이터의 양이 더 적습니다.
- 유료 클라우드를 사용하므로, 과도한 학습 비용을 방지하기 위해서 학습데이터를 적게 설정했습니다.
- 분할된 데이터의 인덱스를 저장할 train_data와 test_data라는 빈 리스트를 생성합니다.


5. 타입별 데이터 분할

- 데이터셋의 균형을 유지하면서 학습용과 테스트용 데이터를 분리하는 부분입니다.
- 각 타입별로 동일한 비율로 분할하는 이유는, 무작위로 전체 데이터를 나누면 특정 타입의 데이터가 한쪽으로 쏠릴 수 있기 때문입니다.


6. OpenAI 형식으로 데이터 변환 함수 정의

- OpenAI 형식으로 데이터를 변환하는 format_data() 함수를 정의합니다.
- message라는 리스트 안에 각각의 대화를 역할과 내용으로 구분하여 담는 구조입니다.


7. 분할된 데이터를 OpenAI 형식으로 변환

- 앞서 분할한 train_data와 test_data의 각 샘플에 format_data() 함수를 적용하여 최종 데이터셋을 생성합니다.
- 각 인덱스에 해당하는 데이터를 format_data() 함수를 이용해 OpenAI 형식으로 변환하여 train_dataset과 test_dataset에 저장합니다.


8. 최종 데이터셋 크기 출력

- 최종적으로 만들어진 train_dataset과 test_dataset의 크기를 출력합니다. 이를 통해 데이터 분할이 의도한 대로 이루어졌는지 확인할 수 있습니다.


9. 분할된 데이터 타입별 분포 출력

- 분할된 데이터의 type별 분포를 출력합니다. 학습 데이터와 테스트 데이터 각각에 대해 type별 개수를 계산하고 출력합니다.
- 이를 통해 데이터 분할이 각 type에 대해 균형 있게 이루어졌는지 검증할 수 있습니다.


### OpenAI 형식 확인하기

- 임의로 345번 샘플을 출력해봅니다.


In [3]:
train_dataset[345]["messages"]

[{'role': 'system',
  'content': '당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.\n\n다음의 지시사항을 따르십시오.\n1. 질문과 검색 결과를 바탕으로 답변하십시오.\n2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.\n3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다."라고 답변하신시오.\n4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오.\n  예를 들어 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.\n5. 예를 들어 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]라고 기재하십시오.\n6. 최대한 다수의 문서를 인용하여 답변하십시오.\n\n검색 결과:\n-----\n문서1: LED(발광다이오드) 조명 등을 만드는 동부그룹 계열사 동부라이텍은 일본 요코하마에 LED 라이트 패널(루미시트) 생산공장을 완공, 본격 양산에 들어갔다고 31일 발표했다. 이 공장은 일본 현지 유통사인 테크타이토와 합작해 세운 공장이다. 루미시트는 얇은 종이판 형태의 LED 조명으로, 이 공장에서는 광고 인테리어용 루미시트 4종을 양산한다.동부라이텍은 2008년 캐나다 토론토에 현지 합작법인 DLC를 세워 북미 고급 매장에서 사용하는 진열대용 루미시트를 생산하고 있다. DLC는 올해 상반기에 약 200억원의 매출을 올렸고, 순이익은 최근 수년간 매년 20%씩 증가하고 있다. 요코하마 공장은 캐나다에서의 성공 모델을 일본으로 옮겨온 것이라는 게 회사 측 설명이다. 동부라이텍은 테크타이토와 합작해 지난해 8월 도쿄에 자본금 1억엔 규모의 합작법인 씨엔디라이텍을 설립한 뒤 현지 공장 가동을 준비해 왔다. 동부라이텍은 테크타이토의 일본 내 유통망을 활용해 일본 루미시트 시장에서 점유율을 늘릴 수 있을 것으로 기대하고 있다.'},
 {'rol

- role이 system인 경우, content에는 앞에서 작성한 시스템 프롬프트와 현재 샘플의 검색 결과가 저장되어 있습니다.
- role이 assistant인 경우, content에는 검색 결과와 사용자 질문을 바탕으로 대규모 언어 모델이 답변해야 할 내용이 작성되어 있습니다.
- OpenAI 형식은 학습하기 위한 최종 형식은 아니며, 전처리를 위한 중간 단계 형태 입니다.
- 학습에 사용하는 최종 형식은 뒤에서 다를 대규모 언어 모델의 토크나이저를 통해 한 번 더 전처리를 진행하고 나서 결정됩니다.


### 데이터 타입 변경

- 현재 train_dataset과 test_dataset은 데이터 타입이 리스트입니다.
- 원할하게 학습을 진행하려면 데이터 타입을 Dataset으로 변경해야 합니다.


In [4]:
# 리스트 형태에서 다시 Dataset 형태로 변환
print(type(train_dataset))
print(type(test_dataset))

train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

print(type(train_dataset))
print(type(test_dataset))

<class 'list'>
<class 'list'>
<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


# Qwen 템플릿 이해하기

- 대규모 언어 모델을 학습할 때 반드시 로드해야 할 두가지가 바로 학습할 모델과 해당 모델에 입력할 데이터를 전처리하는 도구인 토크나이저입니다.
- 대규모 언어 모델은 각각 고유한 토크나이저를 갖고 있으므로 반드시 학습할 모델의 토크나이저를 로드해야 합니다.


### 모델과 토크나이저 로드

- 모델을 로드할 때는 AutoModelForCausalLM.from_pretrained()안에 모델 이름을 기재하고
- 토크나이저를 로드할 때는 AutoTokenizer.from_pretrained() 안에 모델 이름을 기재합니다.
- 앞서 말했듯이 각 대규모 언어 모델은 고유한 토크나이저를 갖고 있으므로 두 개의 코드 안에 들어가는 모델이름은 일반적으로 동일합니다.


In [5]:
# 허깅페이스 모델 이름
# model_id = "Qwen/Qwen2-7B-Instruct"
model_id = "Qwen/Qwen2-1.5B-Instruct"

# 모델과 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


- Qwen 2버전 모델은 중국 IT회사가 공개한 모델로 한국어에도 성능이 뛰어납니다.
- 또 다른 모델로는 LLaMA 3.1,3.2,3.3 Gemma 2~3버전등이 있습니다.


### 템플릿 적용

- 토크나이저를 로드하고 나면 이제 Qwen의 챗 템플릿을 적용해야 합니다.
- 챗 템플릿이란 대규모 언어 모델이 학습할 때 사용하는 특정한 대화 형식, 다시 말해 학습 데이터의 특정 형식을 의미합니다.
- 우리가 사용하는 대규모 언어 모델은 이미 학습된 모델이고, 이를 로드하여 우리의 데이터를 이용하여 추가로 파인튜닝하고는 합니다.
- 대규모 언어 모델은 만들어질 당시에 특정 형식에 맞춰 학습된 상태이므로 파인튜닝할 때도 같은 형식을 지켜야 합니다.
- 대규모 언어 모델을 만들 당시, 첫 학습 때 사용한 템플릿과 다른 형식으로 데이터를 가공하여 파인튜닝을 진행하면
- 제대로 된 성능을 내지 못하여 문맥을 제대로 인식하지 못하거나, 예상한 답변을 제대로 생성하지 못할 가능성이 높아집니다.
- 챗 템플릿은 대규모 언어 모델에 따라 사용하는 형식이 다를 수 있습니다.
- Qwen은 다음과 같은 템플릿에 따라 학습된 모델입니다.


<|im_start|>system  
시스템 프롬프트<|im_end|>  
<|im_start|>user  
사용지 프롬프트(사용자의 질문)<|im_end|>  
<|im_start|>assistant  
대규모 언어 모델이 해야 하는 답변<|im_end|>


- Qwen의 챗 템플릿 형식으로 데이터를 가공하려면, 토크나이저의 apply_chat_template() 에 OpenAI 형식으로 가공된 데이터를 넣으면 됩니다.
- 학습 데이터중 0번 샘플을 챗 템플릿으로 가공하고 나서 실제 출력해 보면 다음과 같습니다.


In [6]:
# 템플릿 적용
text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(text)

<|im_start|>system
당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.

다음의 지시사항을 따르십시오.
1. 질문과 검색 결과를 바탕으로 답변하십시오.
2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.
3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다."라고 답변하신시오.
4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오.
  예를 들어 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.
5. 예를 들어 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]라고 기재하십시오.
6. 최대한 다수의 문서를 인용하여 답변하십시오.

검색 결과:
-----
문서1: 콘코바르는 결국 죽게 되는데, 그 곡절은 이러하다. 라긴의 왕 메스 게그러의 뇌를 굳힌 겻을 울라의 코날이 전리품으로 가지고 있었는데, 코나크타의 전사 케트 막 마가크가 이를 훔쳐갔다. 그리고 케트는 무릿매로 메스 게그러의 뇌를 던져 콘코바르의 머리를 맞추었고, 메스 게그러의 뇌가 콘코바르의 머리통 깊숙히 박혀 버렸다. 이 일이 일어난 곳은 우르카르(Urchair)의 발러 아흐(Baile Ath), 곧 오늘날의 웨스트미스 주 호르셀리프라고 한다. 콘코바르의 의사들은 이 이물질을 제거할 수 없었고, 상처를 봉합만 한 뒤 왕에게 흥분하지 않으면 생명을 유지할 수 있다고 말했다. 7년이 평화롭게 흘러간 뒤 콘코바르는 그리스도가 죽었다는 소식을 듣게 되어 분노했고, 뇌가 터져 죽었다. 머리가 터진 자리에서 뿜어져나온 피의 세례를 받은 결과 그는 기독교인이 되었고 그 영혼은 천국으로 갔다. 콘코바르의 죽음에 관한 이 기록은 매우 얄팍한 기독교화가 이루어져 있는데, 한편 노르드 신화의 토르가 흐룽그니르와 싸우다 머리에 숫돌이 박힌 이야기와 유사한 점이 있다. 어쩌면 두 이야기는 하나의 기원을 공

-Qwen의 챗 템플릿에 맞춰 데이터가 가공되었습니다.


# 4.4 로라 학습을 위한 설정값

- LoRA는 거대한 언어 모델을 학습할 때 풀 파인튜닝 보다 연산량과 GPU 메모리 사용량을 줄여 학습할 수 있는 방법입니다.
- 일반적으로 LLM을 파인튜닝 하려면 모델 내의 모든 변수를 업데이트해야 하지만, 연산량이 급격히 증가하고 많은 GPU 메모리를 소비하게 됩니다.
  - 일반적으로 모델 내에 존재하며 학습 시 값이 업데이트되는 변수들을 '가중치'라고 부릅니다.
- LoRA는 이러한 문제를 해결하기 위해 기존 모델의 가중치는 그대로 두고, 추가로 작은 변수를 가진 행렬을 덧붙여 이에 해당하는 추가적인 가중치 행렬만 학습하는 방식을 사용합니다.
- 필요한 가중치만 학습하는 방식입니다.
  - 빠른 학습 속도: 학습해야 하는 변수의 개수가 줄어들어 학습 속도가 빨라집니다.
  - 하드웨어 요구량 감소: 기존 모델을 학습하는 것보다 훨씬 적은 연산량으로도 조정이 가능하여, 많은 양의 GPU리소스가 없더라도 학습할 수 있습니다.


In [7]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

- lora_alpha
  - LoRA 학습이 기존 LLM 예측 결과에 얼마나 영향을 미칠지를 결정하는 값입니다.
  - 값이 클수록 학습한 정보가 더 강하게 반영되고, 값이 작으면 기존 모델의 원래 특성이 더 많이 유지됩니다.
  - 1이면 기존 모델이 거의 그대로 유지되며, LoRA학습 후의 효과가 미미합니다.
  - 100이면 기존 모델보다 LoRA 학습 정보가 더 강하게 적용됩니다.
- lora_dropout
  - 학습할 때 일부 정보를 의도적으로 제외하여, 모델이 특정 데이터에 과도하게 의존하지 않도록 만드는 값입니다.
  - 0.1로 설정하면, 학습 과정에서 일부 정보(약 10%)가 의도적으로 제외된 상태로 학습이 진행됩니다.
  - 특정 데이터에만 최적화되지 않고, 다양한 상황에서도 잘 작동할 수 있도록 학습할 수 있습니다.
  - 즉 새로운 데이터에도 적응할 수 있도록 돕는 역할을 합니다.
- r
  - LoRA 학습에서 학습할 정보의 양을 결정하는 값입니다.
  - LoRA 학습은 기존 모델 전체를 수정하는 것이 아니라, 특정 부분만 선택적으로 학습합니다.
  - 학습할 정보의 크기를 결정하는 값이 r입니다.
  - r 값이 클수록 더 많은 정보를 학습하지만, 그 만큼 메모리 사용량과 연산량이 증가합니다.
  - 값이 작으면 메모리는 절약되지만, 학습범위도 줄어듭니다.
  - r=2이면 LoRA 학습이 모델을 매우 작은 범위에서만 조정하고, r=64면 모델을 더 넓은 범위에서 조정할 수 있습니다.
- bias
  - LoRA가 학습하는 과정에서 LLM의 편향값을 조정할지를 결정하는 값입니다.
  - LLM은 입력 처리하는 과정에서 출력값을 조정하는 요소 중 하나로 편향값을 포함합니다.
  - LoRA는 기본적으로 이러한 편향값을 변경하지 않지만, 필요에 따라 LoRA가 편향값까지 학습하도록 설정할 수 있습니다.
  - none은 기존 모델의 편향값을 조정하지 않는 설정입니다. all을 선택하면 기존 모델의 편향값까지 LoRA학습으로 조정하여 더 큰 변화를 줄 수 있습니다.
- target_modules
  - LoRA 학습을 적용할 특정 부분을 선택하는 값입니다.
  - 모델은 여러 개의 단계로 이루어져 있습니다. 모든 부분을 LoRA 방식으로 학습하는 것이 아니라 필요한 부분만 선택적으로 학습할 수 있습니다.
  - q_proj, v_proj는 모델이 입력을 처리하는 과정에서 중요한 역할을 하는 부분입니다.
  - 이 부분을 LoRA방식으로 학습하면 기존보다 더 유연하게 작동할 수 있도록 조정할 수 있습니다.
- task_type
  - LoRA 학습이 적용될 모델의 작업 유형을 지정하는 값입니다.
  - 모델이 어떤 방식으로 동작하는지에 따라 LoRA 학습 방식도 달라지므로 정확하게 설정해야 합니다.
  - "CAUSAL_ML"은 입력된 텍스트를 기반으로 다음 단어를 예측하는 모델에 사용됩니다.
  - 이는 LLM과 같이 왼쪽에서 오른쪽으로 순차적으로 문장을 생성하는 모델에 적합합니다.


# 학습을 위한 설정값

- 모델을 학습할 때 필요한 다양한 설정값을 정의하는 도구인 SFTConfig를 설정합니다.
- SFTConfig는 학습과정에서 모델이 어떻게 업데이트될지를 조정하는 값입니다.
- LoRA와 연관된 설정들은 LoRAConfig에서 관리하지만, 그 외에 학습전반에 걸친 설정은 SFTConfig에서 관리합니다.


In [8]:
args = SFTConfig(
    output_dir="qwen2-1.5b-rag-ko",  # 저장될 디렉터리와 저장소 ID
    num_train_epochs=3,  # 학습할 총 에포크 수
    per_device_eval_batch_size=2,  # GPU당 배치 크기
    gradient_accumulation_steps=2,  # 그래디언트 누적 스텝 수
    gradient_checkpointing=True,  # 메모리 절약을 위한 체크포인팅
    optim="adamw_torch_fused",  # 최적화기
    logging_steps=10,  # 로깅 기록 주기
    save_strategy="steps",  # 저장 전략
    save_steps=50,  # 저장 주기
    bf16=True,  # bfloat16 사용
    learning_rate=1e-4,  # 학습률
    max_grad_norm=0.3,  # 그래디언트 클리핑
    warmup_ratio=0.03,  # 워밍업 비율
    lr_scheduler_type="constant",  # 고정 학습률
    push_to_hub=False,  # 허브 업로드 안함
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=None,
)

average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.


- output_dir
  - 학습된 모델을 저장할 위치를 지정합니다.
- num_train_epochs
  - 학습할 총 횟수입니다. num_train_epochs = 3이면, 학습데이터를 3번 반복해 학습합니다.
  - 데이터를 여러 번 학습할수록 더 많이 배울 수 있지만, 너무 반복하면 기존 학습 데이터에 지나치게 맞춰져 새로운 데이터에서 성능이 떨어질 수 있습니다.
- per_device_train_batch_size
  - 한 번 학습할 때 처리하는 데이터 개수입니다.
  - per_device_train_batch_size=2이면, 한번에 2개의 데이터를 사용해서 학습합니다.
  - 학습할 때 데이터를 하나씩 처리하는 것이 아니라 여러 개를 묶어서 한 번에 학습하는데, 이를 배치라고 합니다.
  - 배치 크기가 크면 학습 속도가 빨라질 수 있지만, 메모리를 많이 사용합니다.
- gradient_accmulation_steps
  - 여러 번 계산한 결과를 모아서 한 번에 적용하는 기능입니다.
  - gradient_accumulation_steps=2이면, 2번 계산한 결과를 모아서 한번에 모델을 업데이트합니다.
  - 메모리가 부족할 때 작읍 배치를 여러 번 모아서 학습할 수 있도록 돕는 기능입니다.
- gradient_checkpointing
  - 메모리를 절약하는 기능입니다.
  - gradient_checkpointing=True이면, 학습할 때 일부 정보를 저장하지 않고 필요할 때 다시 계산하여 메모리를 절약하는 기능이 활성화됩니다.
  - 속도가 조금 느려질 수 있지마느 메모리를 적게 쓰는 장점이 있습니다.
- optim
  - 모델을 업데이트할 때 사용하는 최적화 방법입니다.
  - "adamw_torch_fused"는 AdamW 최적화 기법을 사용하여 모델을 업데이트 하는 방식입니다.
  - AdamW는 안정적인 학습이 가능하도록 제안된 학습 방식입니다.
- logging_steps
  - 학습 중 진행상황을 얼마나 자주 기록할지 결정하는 값입니다.
  - logging_steps=10이면, 10번의 학습 스텝마다 진행상황을 기록합니다.
  - 너무 자주 기록하면 속도가 느려질 수 있고, 너무 드물면 학습 상태를 파악하기 어려울 수 있습니다.
- save_strategy 및 save_steps
  - 모델을 저장하는 방식과 주기입니다.
  - save_strategy = "steps"이면 지정된 스텝마다 모델을 저장합니다.
  - save_steps=50이면, 50번의 학습 스텝마다 모델이 저장됩니다.
  - 모델을 중간에 저장하지 않으면 학습 중 문제가 발생할 경우 처음부터 다시 학습해야 할 수도 있기 때문에 적절한 주기로 저장하는 것이 중요합니다.
- bf16
  - bfloat16(16비트 부동소수점) 사용 여부를 지정합니다.
  - bf16=True 이면 bfloat16 형식을 사용하여 모델을 학습합니다.
  - bfloat16은 메모리를 절약하면서도 연산의 정확도를 유지할 수 있는 데이터 형식으로 NVIDIA A100과 같은 최신 GPU에서 성능을 최적화할 수 있습니다.
- learning_rate
  - 하급 속도를 조절하는 값입니다.
  - learning_rate=1e-4이면, 학습 속도를 0.0001로 설정합니다.
  - 값이 너무 크면 모델이 불안정하게 학습될 수 있고, 값이 너무 작으며 학습이 느려질 수 있습니다.
- max_grad_norm
  - 그래디언트 클리핑 설정입니다.
  - max_grad_norm=0.3이면, 모델이 한 번 업데이트 될때 변화량을 제한하여 너무 급격한 변화가 일어나지 않도록 조정합니다.
  - 변화량이 너무 커지는 경우 모델이 불안정해질 수 있기 때문에 이를 제한하는 기능입니다.
- warmup_ratio
  - 학습 초반에 천천히 시작하도록 조정하는 값입니다.
  - warmup_ratio=0.03이면 초반 3% 구간동안 학습률을 서서히 증가시키며 적응하는 방식으로 학습을 진행합니다.
  - 처음부터 너무 빠르게 학습하면 모델이 불안정할 수 있기 때문에, 일정 구간 동안 천천히 학습 속도를 올려 안정적으로 학습할 수 있도록 합니다.
- lr_scheduler_type
  - 학습률을 어떻게 조정할지 결정합니다.
  - lr_scheduler_type="constant"이면, 학습률을 일정하게 유지하는 방식으로 학습합니다.
  - 일반적으로 학습이 진행됨에 따라 학습속도를 줄이는 방식도 있습니다.
- push_to_hub
  - 학습된 모델을 허깅페이스 웹사이트에서 접근 가능한 모델 저장소에 업로드할지 여부입니다.
  - push_to_hub=False이면, 학습된 모델을 업로드하지 않습니다.
  - 허킹페이스 모델 저자오에 자동으로 업로드하려면 True로 설정해야 합니다.
- remove_unused_columns
  - 불필요한 컬럼 제거 여부입니다. remove_unused_columns=False리면, 데이터에서 사용하지 않는 컬럼을 자동으로 제거하지 않습니다.
  - 이 값을 True로 설정하면, 모델이 사용하지 않는 데이터를 자동으로 정리하여 학습을 최적화할 수 있습니다.
- dataset_kwargs
  - 데이터셋 관련 추가 설정입니다.
  - dataset_kwargs = {"skip_prepare_dataset": True}이면, 데이터를 미리 준비하는 과정을 생략합니다.
  - 데이터셋이 이미 준비되어 있는 경우, 불필요한 처리를 생략하여 속도를 높일수 있습니다.
- report_to
  - 학습과정정보를 어디로 보고할지 지정하는 값입니다.
  - report_to = None이면 학습과정을 별도로 로그를 저장하는 툴에 기록하지 않습니다.


# 4.6 정수 인코딩

- LLM에 학습 데이터를 전달하기 전에 수치화 작업을 거칩니다.
- 이 과정을 인코딩이라 부르며 앞서 로드한 토크나이저를 통해 수행합니다.


### 대규모 언어 모델 학습용 데이터 구성과 인코딩 방법

- LLM 모델을 학습할 때는 데이터의 입력과 출력, 두 부분이 필요합니다.

  - 시스템 프롬프트: 당신은 친절한 AI 어시스턴트입니다.
  - 사용자 프롬프트: 안녕하세요, 오늘의 날씨는 어떤가요?
  - 모델의 응답: 안녕하세요! 오늘 날씨는 맑고 화창합니다.

- 이 데이터를 학습하려면 먼저 앞서 진행한 바와 같이 챗 템플릿을 적용해야 합니다.

```
<!im_start|>system
당신은 친절한 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
안녕하세요, 오늘 날씨는 어떤가요?<|im_end|>
<|im_start|>assistant
안녕하세요! 오늘 날씨는 맑고 화창합니다.<|im_end|>
```

- 챗 템플릿을 적용하고 나서 이로부터 input_ids와 labels를 생성합니다.
- input_ids는 전체 텍스트를 토크나이저를 사용해 정수로 인코딩한 결과입니다.
- 각 토큰이 어떤 정수로 변환되는지와 같은 정보는 앞서 로드한 토크나이저를 통해 얻을 수 있습니다.
- Qwen 모델의 토크나이저로 인코딩 작업을 통해 정수로 변환한 input_ids는 다음과 같습니다.


```
input_ids = [
    151644, 198,
    8948, 198, 64795, 82528, 33704, 90711, 250, 126550,
    198, 151645, 198,
    ...
    239, 34395, 46832, 242, 130095, 60838, 13,
    198, 151645, 198
]
```

- 각 토큰이 어떤 정수로 매핑되는지는 LLM모델마다 전부 다를 수 있습니다.
- 모델이 학습해야 하는 부분은 시스템 프롬프트와 사용자 프롬프트를 보고 적절한 응답을 생성해야 하는 것입니다.
- labels는 input_ids에서 모델이 직접 생성할 필요가 없는 부분을 -100으로 처리합니다.

```
labels = [
    -100, -100
    -100, -100, -100, -100, -100, -100, -100, -100,
    -100, -100, -100
    ...
    239, 34395, 46832, 242, 130095, 60838, 13,
    -100, -100, -100

]
```

- 이렇게 설정하면 모델이 labels에서 -100이 아닌 부분만 학습하여 LLM이 생성해야 하는 응답만 학습하도록 유도할 수 있습니다.
- 모델이 불필요한 부분을 학습하려 하지 않고 올바른 답변을 생성하는데 집중할 수 있도록 하는 기본적인 학습 구조입니다.


In [ ]:
def collate_fn(batch):
    new_batch = {"input_ids": [], "attention_mask": [], "labels": []}

    for example in batch:
        # messages의 각 내용에서 개행문자 제거
        clean_messages = []
        for message in example["messages"]:
            clean_message = {"role": message["role"], "content": message["content"]}
            clean_messages.append(clean_message)

        # 깨끗해진 메시지로 템플릿 적용
        text = tokenizer.apply_chat_template(
            clean_messages, tokenize=False, add_generation_prompt=False
        ).strip()

        # 텍스트를 토큰화
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        # 레이블 초기화
        labels = [-100] * len(input_ids)

        # assistant 응답 부분 찾기
        im_start = "<|im_start|>"
        im_end = "<|im_end|>"
        assistant = "assistant"

        # 토큰 ID 가져오기
        im_start_tokens = tokenizer.encode(im_start, add_special_tokens=False)
        im_end_tokens = tokenizer.encode(im_end, add_special_tokens=False)
        assistant_tokens = tokenizer.encode(assistant, add_special_tokens=False)

        i = 0
        while i < len(input_ids):
            # <|im_start|>assistant 찾기
            if (
                i + len(im_start_tokens) <= len(input_ids)
                and input_ids[i : i + len(im_start_tokens)] == im_start_tokens
            ):
                # assistant 토큰 찾기
                assistant_pos = i + len(im_start_tokens)
                if (
                    assistant_pos + len(assistant_tokens) <= len(input_ids)
                    and input_ids[assistant_pos : assistant_pos + len(assistant_tokens)]
                    == assistant_tokens
                ):
                    # assistant 응답의 시작 위치로 이동
                    current_pos = assistant_pos + len(assistant_tokens)

                    # <|im_end|>를 찾을 때까지 레이블 설정
                    while current_pos < len(input_ids):
                        if (
                            current_pos + len(im_end_tokens) <= len(input_ids)
                            and input_ids[
                                current_pos : current_pos + len(im_end_tokens)
                            ]
                            == im_end_tokens
                        ):
                            # <|im_end|> 토큰도 레이블에 포함
                            for j in range(len(im_end_tokens)):
                                labels[current_pos + j] = input_ids[current_pos + j]
                            break
                        labels[current_pos] = input_ids[current_pos]
                        current_pos += 1
                    i = current_pos
            i += 1
        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)

    # 패딩 적용
    max_length = max(len(ids) for ids in new_batch["input_ids"])
    for i in range(len(new_batch["input_ids"])):
        padding_length = max_length - len(new_batch["input_ids"][i])

        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * padding_length)
        new_batch["attention_mask"][i].extend([0] * padding_length)
        new_batch["labels"][i].extend([-100] * padding_length)

    # 텐서로 변환
    for k, v in new_batch.items():
        new_batch[k] = torch.tensor(v)

    return new_batch


- collate_fn(batch) 함수는 자연어 처리 모델 학습에 필요한 데이터를 전처리하는 역할을 수행합니다.
- 해당 함수 내에서 input_ids는 전체 대화를 숫자로 바꾼 결과물입니다.
- 토크나이저는 "<|im_start|>"와 "<|im_end|>" 같은 특수 토큰을 포함한 모든 텍스트를 숫자로 변환합니다.
- im_start_tokens와 assistant_tokens를 먼저 인코딩해서 이 토큰들의 숫자값을 미리 준비해둡니다.


- labels는 모델이 실제로 생성해내야 할 목표값입니다.
- 이 코드의 핵심은 assistant가 답변한 부분만 골라서 학습시키는 것입니다.
- assistant가 답변한 부분은 input_ids의 값을 그대로 labels에 복사하고, 나머지는 전부 -100으로 채웁니다.
- -100dms pytorch에서 학습할 때 무시하게 되어 있는 특별한 값입니다.


- 코드를 보면 while 문을 사용해서 input_ids 안에서 "<|im_start|>assistant"로 시작하는 부분을 찾습니다.
- 이 부분부터 "<|im_end|>"가 나올때까지가 assistant의 응답입니다.
- 이 구간의 토큰들만 labels에 복사하고 나머지는 전부 -100을 넣습니다.
- 이렇게 하면 모델은 assistant의 응답만 학습하게 됩니다.


- 이제 임의의 샘플에 대해 실제로 전처리가 제대로 진행되는지 확인해봅시다.
- 학습 데이터 중 0번 인덱스를 가진 첫번째 샘플에 대해 collate_fn 함수를 적용하여 결과를 확인합니다.


In [18]:
# 데이터의 최대 길이 한도를 지정. 최대 8192개의 토큰까지만 사용
max_seq_length = 8192

example = train_dataset[0]
batch = collate_fn([example])

print("입력에 대한 정수 인코딩 결과:")
print(batch["input_ids"][0].tolist())
print("레이블에 대한 정수 인코딩 결과:")
print(batch["labels"][0].tolist())

[151645]
입력에 대한 정수 인코딩 결과:
[151644, 8948, 198, 64795, 82528, 33704, 85322, 77226, 98801, 18411, 81718, 144059, 42039, 138520, 19391, 143604, 129264, 130650, 382, 13146, 48431, 20401, 66790, 29326, 131193, 17877, 125686, 125548, 139713, 624, 16, 13, 138520, 53680, 85322, 77226, 98801, 18411, 81718, 144059, 42039, 143604, 16186, 139713, 624, 17, 13, 85322, 77226, 98801, 19391, 130768, 130213, 17877, 143604, 16186, 125476, 34395, 53900, 21329, 95577, 139713, 624, 18, 13, 138520, 19391, 128605, 143603, 12802, 85322, 77226, 98801, 19391, 130671, 32290, 85322, 77226, 98801, 126377, 330, 33883, 64795, 138520, 93, 19391, 128605, 130213, 12802, 136673, 1189, 129254, 143604, 135784, 29326, 57268, 624, 19, 13, 143604, 47836, 53618, 142976, 139236, 18411, 142616, 82190, 53435, 40853, 129549, 53435, 125068, 17877, 140174, 128836, 32290, 5140, 240, 97, 19391, 36330, 250, 125746, 16560, 23084, 126402, 83634, 17380, 94613, 139236, 84621, 47324, 18411, 129624, 20487, 139713, 624, 220, 95617, 18411, 129

- 출력 결과를 살펴보면 입력에 대한 정수 인코딩 결과와 레이블에 대한 정수 인코딩 결과의 길이가 같습니다.
- 레이블에 대한 정수 인코딩 결과에서 LLM모델의 실제 답변 부분을 제외하고는 전부 -100으로 채워진 값이 출력됩니다.


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    max_seq_length=max_seq_length,  # 최대 시퀀스 길이 설정
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

# 학습 시작
# trainer.train() #모델이 자동으로 허브와 output_dir에 저장됨

# 모델 저장
# trainer.save_model()  #최종 모델을 저장


- 약 30분 정도의 시간이 지나 학습이 끝났습니다.
